# 勇者傳說 — Building V2 Re-generator (T4 GPU)

**Auto-detect** low-quality building sprites from V1 output and **re-generate** with hardened prompts.

Key improvements over V1:
- Auto-scans V1 output for quality failures (opaque%, bbox%, file size)
- Stricter quality thresholds per building type (castles need more coverage than houses)
- More retries (5) with varied seeds
- Keeps best attempt even if all fail threshold
- Per-type LoRA weight adjustment

**Self-contained**: No repo clone needed.  
**Output**: Google Drive `/MyDrive/ai-rpg-game/outputs/`  
**Resume**: 中斷後重跑自動跳過已完成的

In [1]:
!pip install -q "numpy<2" scipy
!pip install -q diffusers transformers accelerate safetensors "Pillow>=10,<12" rembg onnxruntime

import gc, json, os, time
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

GDRIVE_BASE = Path('/content/drive/MyDrive/ai-rpg-game')
OUTPUT_BASE = GDRIVE_BASE / 'outputs' if ON_COLAB else Path('outputs')
MODELS_DIR = GDRIVE_BASE / 'models' if ON_COLAB else Path('models')

for d in [OUTPUT_BASE, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
    DEVICE = 'cuda'
    DTYPE = torch.float16
    print(f'GPU: {name} ({vram:.1f} GB VRAM)')
else:
    DEVICE = 'cpu'
    DTYPE = torch.bfloat16
    print('WARNING: No GPU!')

print(f'Output: {OUTPUT_BASE}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but 

In [2]:
from PIL import Image
import numpy as np


class ProgressTracker:
    def __init__(self, task_name, output_dir):
        self.file = Path(output_dir) / f'_progress_{task_name}.json'
        self.completed = set()
        self.start_time = time.time()
        if self.file.exists():
            try:
                data = json.loads(self.file.read_text())
                self.completed = set(data.get('completed', []))
                print(f'[resume] {len(self.completed)} items already done')
            except Exception:
                pass

    def is_done(self, name): return name in self.completed

    def mark_done(self, name):
        self.completed.add(name)
        self.file.parent.mkdir(parents=True, exist_ok=True)
        self.file.write_text(json.dumps({
            'completed': sorted(self.completed),
            'count': len(self.completed),
            'last_updated': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, indent=2))

    def summary(self, total):
        done = len(self.completed)
        elapsed = time.time() - self.start_time
        if done > 0:
            remaining = (total - done) * (elapsed / done)
            return f'{done}/{total} done, ~{remaining/60:.0f} min remaining'
        return f'0/{total} done'


def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def print_vram():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated(0) / 1024**3
        props = torch.cuda.get_device_properties(0)
        total = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
        print(f'[vram] {used:.1f}/{total:.1f} GB')


_rmbg_model = None
_rembg_session = None


def remove_background(img):
    """Remove background: RMBG-2.0 primary, rembg fallback."""
    global _rmbg_model, _rembg_session
    try:
        if _rmbg_model is None:
            from transformers import pipeline
            _rmbg_model = pipeline('image-segmentation',
                                   model='briaai/RMBG-2.0',
                                   trust_remote_code=True,
                                   device=0 if torch.cuda.is_available() else -1)
        result = _rmbg_model(img)
        mask = result[0]['mask'] if isinstance(result, list) else result['mask']
        img = img.convert('RGBA')
        img.putalpha(mask.convert('L'))
        return img
    except Exception as e:
        print(f'[warn] RMBG-2.0 failed: {e}, trying rembg...')
    try:
        from rembg import remove, new_session
        if _rembg_session is None:
            _rembg_session = new_session('u2net')
        return remove(img, session=_rembg_session)
    except Exception as e:
        print(f'[warn] rembg failed: {e}')
        return img


# ─── Per-type quality thresholds ───
QUALITY_THRESHOLDS = {
    'castle':  {'min_opaque': 0.20, 'min_bbox': 0.40},
    'gate':    {'min_opaque': 0.15, 'min_bbox': 0.25},
    'inn':     {'min_opaque': 0.10, 'min_bbox': 0.25},
    'shop':    {'min_opaque': 0.10, 'min_bbox': 0.25},
    'church':  {'min_opaque': 0.10, 'min_bbox': 0.25},
    'house':   {'min_opaque': 0.10, 'min_bbox': 0.20},
}


def quality_check(img, btype='house'):
    """Check building quality with per-type thresholds."""
    thresholds = QUALITY_THRESHOLDS.get(btype, QUALITY_THRESHOLDS['house'])
    arr = np.array(img.convert('RGBA'))
    alpha = arr[:, :, 3]
    total = alpha.size
    opaque_pct = np.sum(alpha > 10) / total

    if opaque_pct < thresholds['min_opaque']:
        return False, opaque_pct

    rows = np.any(alpha > 10, axis=1)
    cols = np.any(alpha > 10, axis=0)
    if not rows.any():
        return False, 0.0
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    bbox_pct = ((rmax - rmin + 1) * (cmax - cmin + 1)) / total

    if bbox_pct < thresholds['min_bbox']:
        return False, opaque_pct

    return True, opaque_pct


def scan_buildings_quality(bld_dir):
    """Scan existing buildings for quality failures. Returns list of failed filenames (no ext)."""
    fails = []
    for png in sorted(Path(bld_dir).glob('*.png')):
        if png.stem.startswith('_'):
            continue
        img = Image.open(png).convert('RGBA')
        arr = np.array(img)
        w, h = img.size
        alpha = arr[:, :, 3]
        total = alpha.size
        opaque_pct = np.sum(alpha > 10) / total

        # Detect building type from filename
        btype = 'house'
        for t in ['castle', 'gate', 'inn', 'shop', 'church']:
            if t in png.stem:
                btype = t
                break

        passed, _ = quality_check(img, btype)
        kb = png.stat().st_size / 1024
        status = 'OK' if passed else 'FAIL'

        rows = np.any(alpha > 10, axis=1)
        cols = np.any(alpha > 10, axis=0)
        bbox_pct = 0.0
        if rows.any():
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            bbox_pct = ((rmax - rmin + 1) * (cmax - cmin + 1)) / total

        print(f'  {png.stem:<35} {w}x{h}  {opaque_pct:>6.1%}  bbox={bbox_pct:>5.1%}  {kb:>5.0f}KB  {status}')
        if not passed:
            fails.append(png.stem)
    return fails

In [3]:
# ─── V2 Prompt Data ───
# Same regions + building types as V1, with hardened prompts for known issues.

REGIONS = [
    {'id': 'r1', 'name': 'region_hero', 'theme': 'classic medieval European kingdom', 'elements': 'stone and timber walls, blue banners, peaked slate roofs, iron-bound doors'},
    {'id': 'r2', 'name': 'region_elf', 'theme': 'elven forest kingdom', 'elements': 'living wood structure, golden filigree, green vine decorations, leaf-shaped windows'},
    {'id': 'r3', 'name': 'region_treant', 'theme': 'ancient treehouse village', 'elements': 'hollow tree trunk walls, moss covering, mushroom platforms, root foundations'},
    {'id': 'r4', 'name': 'region_beast', 'theme': 'tribal savanna settlement', 'elements': 'mud-brick and clay walls, animal pelt awnings, bone decorations, thatched roofs'},
    {'id': 'r5', 'name': 'region_merfolk', 'theme': 'coral aquatic kingdom', 'elements': 'shell and coral architecture, blue-white palette, pearl accents, wave patterns'},
    {'id': 'r6', 'name': 'region_giant', 'theme': 'massive stone fortress', 'elements': 'oversized proportions, grey granite blocks, iron rivets, heavy stone archways'},
    {'id': 'r7', 'name': 'region_dwarf', 'theme': 'underground forge city', 'elements': 'carved stone facade, forge glow from windows, copper and bronze trim, anvil motifs'},
    {'id': 'r8', 'name': 'region_undead', 'theme': 'gothic dark castle ruins', 'elements': 'purple glow, cobwebs, gargoyles, cracked dark stone, eerie green lights'},
    {'id': 'r9', 'name': 'region_volcano', 'theme': 'obsidian volcanic tribal', 'elements': 'basalt walls, lava glow cracks, fire motifs, dark stone with ember accents'},
    {'id': 'r10', 'name': 'region_hotspring', 'theme': 'Japanese onsen town', 'elements': 'wooden beams, bamboo accents, paper lanterns, sliding doors, curved roofs'},
    {'id': 'r11', 'name': 'region_mountain', 'theme': 'alpine mountain lodge', 'elements': 'snow-capped roof, prayer flags, heavy timber, stone chimney, warm glow'},
    {'id': 'r12', 'name': 'region_demon', 'theme': 'dark demonic fortress', 'elements': 'bone and skull motifs, red banners, demonic carvings, dark iron spikes'},
]

BUILDING_TYPES = [
    {'type': 'castle', 'prefix': 'bld_castle', 'desc': 'grand royal castle with towers and main gate, imposing multi-story palace, large solid structure filling most of the frame', 'gen_w': 512, 'gen_h': 512, 'final_w': 320, 'final_h': 320, 'steps': 28, 'guidance': 8.0, 'lora_weight': 0.6},
    {'type': 'gate', 'prefix': 'bld_gate', 'desc': 'fortified entrance gate archway with portcullis, wide stone gatehouse, solid structure', 'gen_w': 384, 'gen_h': 256, 'final_w': 192, 'final_h': 128, 'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8},
    {'type': 'inn', 'prefix': 'bld_inn', 'desc': 'cozy inn tavern building with hanging lantern sign, warm light from windows, solid two-story building', 'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128, 'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8},
    {'type': 'shop', 'prefix': 'bld_shop', 'desc': 'merchant shop building with display window and goods, shop awning, solid two-story structure', 'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128, 'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8},
    {'type': 'church', 'prefix': 'bld_church', 'desc': 'sacred church temple building with steeple cross bell tower, stained glass window, solid stone structure', 'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128, 'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8},
    {'type': 'house', 'prefix': 'bld_region', 'desc': 'residential house dwelling, solid home with door and windows, two-story building', 'gen_w': 256, 'gen_h': 256, 'final_w': 128, 'final_h': 128, 'steps': 25, 'guidance': 7.5, 'lora_weight': 0.8},
]

META = {
    'style_prefix_xl': '16-bit JRPG pixel art, top-down RPG building sprite',
    'style_suffix_xl': 'front view, single building centered, transparent background, clean edges, detailed pixel art',
    'negative_prompt_xl': 'blurry, modern, multiple buildings, text, UI, photograph, realistic photo, people, characters, watermark',
}

# Build lookup: name -> entry
ALL_ENTRIES = {}
for region in REGIONS:
    for btype in BUILDING_TYPES:
        name = f"{btype['prefix']}_{region['id']}"
        ALL_ENTRIES[name] = {
            'name': name,
            'prompt_xl': f"{region['theme']} {btype['desc']}, {region['elements']}",
            'gen_w': btype['gen_w'], 'gen_h': btype['gen_h'],
            'final_w': btype['final_w'], 'final_h': btype['final_h'],
            'steps': btype['steps'], 'guidance': btype['guidance'],
            'lora_weight': btype['lora_weight'], 'type': btype['type'],
            'region': region['name'],
        }

print(f'Total prompt entries: {len(ALL_ENTRIES)}')

Total prompt entries: 72


In [4]:
# ─── Auto-detect failures from V1 output ───
bld_dir = OUTPUT_BASE / 'buildings'

if bld_dir.exists() and any(bld_dir.glob('*.png')):
    print('=== Scanning V1 output for quality failures ===')
    auto_fails = scan_buildings_quality(bld_dir)
    print(f'\nAuto-detected failures: {len(auto_fails)}')
    for f in auto_fails:
        print(f'  {f}')
else:
    print('No V1 output found — will generate all entries')
    auto_fails = list(ALL_ENTRIES.keys())

# Also include any entries whose files don't exist
missing = [name for name in ALL_ENTRIES if not (bld_dir / f'{name}.png').exists()]
if missing:
    print(f'\nMissing files: {len(missing)}')
    for m in missing:
        print(f'  {m}')

# Combine
to_regen = sorted(set(auto_fails + missing))
print(f'\nTotal to (re)generate: {len(to_regen)}')

if len(to_regen) == 0:
    print('\nAll buildings pass quality check! Nothing to regenerate.')

=== Scanning V1 output for quality failures ===
  bld_blacksmith                      64x64   40.8%  bbox=42.5%      4KB  OK
  bld_castle                          64x64   23.3%  bbox=40.9%      3KB  OK
  bld_castle_r1                       320x320   66.7%  bbox=98.4%    159KB  OK
  bld_castle_r10                      320x320   71.4%  bbox=98.1%    152KB  OK
  bld_castle_r11                      320x320   67.9%  bbox=96.9%    148KB  OK
  bld_castle_r12                      320x320   70.7%  bbox=99.7%    163KB  OK
  bld_castle_r2                       320x320   71.8%  bbox=98.8%    160KB  OK
  bld_castle_r3                       320x320   28.9%  bbox=83.4%     70KB  OK
  bld_castle_r4                       320x320   47.0%  bbox=96.2%    113KB  OK
  bld_castle_r5                       320x320   67.2%  bbox=96.6%    145KB  OK
  bld_castle_r6                       320x320   75.1%  bbox=97.2%    166KB  OK
  bld_castle_r7                       320x320   62.0%  bbox=95.6%    157KB  OK
  bld_ca

In [ ]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if len(to_regen) == 0:
    print('Nothing to regenerate — skipping pipeline load.')
    pipe = None
else:
    SDXL_HUB = 'stabilityai/stable-diffusion-xl-base-1.0'
    LORA_HUB = 'nerijs/pixel-art-xl'

    sdxl_cache = MODELS_DIR / 'stable-diffusion-xl-base-1.0'
    model_path = str(sdxl_cache) if (sdxl_cache / 'model_index.json').exists() else SDXL_HUB

    print(f'[pipeline] Loading SDXL ({DEVICE})...')
    t0 = time.time()

    kwargs = {'torch_dtype': DTYPE, 'use_safetensors': True, 'low_cpu_mem_usage': False}
    if 'stabilityai' in model_path:
        kwargs['variant'] = 'fp16'

    pipe = StableDiffusionXLPipeline.from_pretrained(model_path, **kwargs)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config,
        algorithm_type='dpmsolver++',
        use_karras_sigmas=True,
    )

    try:
        pipe.load_lora_weights(LORA_HUB, adapter_name='pixel_xl')
        pipe.set_adapters(['pixel_xl'], adapter_weights=[0.8])
        print('[pipeline] pixel-art-xl LoRA loaded (weight=0.8)')
    except Exception as e:
        print(f'[warn] LoRA failed: {e}')

    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()

    if 'stabilityai' in model_path and ON_COLAB:
        try:
            pipe.save_pretrained(str(sdxl_cache))
            print(f'[pipeline] Cached to {sdxl_cache}')
        except Exception:
            pass

    print(f'[pipeline] Ready in {time.time()-t0:.1f}s')
    print_vram()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Nothing to regenerate — skipping pipeline load.


In [6]:
def postprocess_building(img, entry):
    """Remove background, crop, resize to final building size."""
    final_w = entry['final_w']
    final_h = entry['final_h']
    img = remove_background(img.convert('RGBA'))

    arr = np.array(img)
    alpha = arr[:, :, 3]
    rows = np.any(alpha > 10, axis=1)
    cols = np.any(alpha > 10, axis=0)
    if not rows.any():
        return img.resize((final_w, final_h), Image.NEAREST)

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    cropped = img.crop((cmin, rmin, cmax + 1, rmax + 1))

    cw, ch = cropped.size
    scale = min(final_w / cw, final_h / ch)
    new_w = max(1, int(cw * scale))
    new_h = max(1, int(ch * scale))
    resized = cropped.resize((new_w, new_h), Image.NEAREST)

    canvas = Image.new('RGBA', (final_w, final_h), (0, 0, 0, 0))
    x = (final_w - new_w) // 2
    y = final_h - new_h
    canvas.paste(resized, (x, y))
    return canvas


# ─── Re-generate failed/missing buildings ───
if pipe is None:
    print('No pipeline loaded — nothing to do.')
else:
    out_dir = OUTPUT_BASE / 'buildings'
    out_dir.mkdir(parents=True, exist_ok=True)

    tracker = ProgressTracker('buildings_v2', OUTPUT_BASE)
    regen_entries = [ALL_ENTRIES[n] for n in to_regen if n in ALL_ENTRIES]
    remaining = sum(1 for e in regen_entries if not tracker.is_done(e['name']))
    print(f'\nV2 Re-generation: {len(regen_entries)} entries, {remaining} remaining')

    MAX_RETRIES = 5

    for i, entry in enumerate(regen_entries):
        name = entry['name']
        out_path = out_dir / f'{name}.png'

        if tracker.is_done(name):
            print(f'  [skip] {name}')
            continue

        prompt = f"{META['style_prefix_xl']}, {entry['prompt_xl']}, {META['style_suffix_xl']}"
        neg = META['negative_prompt_xl']
        btype = entry['type']

        # Per-type LoRA weight
        pipe.set_adapters(['pixel_xl'], adapter_weights=[entry['lora_weight']])

        print(f"[{i+1}/{len(regen_entries)}] {name} ({btype}, {entry['gen_w']}x{entry['gen_h']}→{entry['final_w']}x{entry['final_h']})")
        t0 = time.time()

        best_img = None
        best_opaque = 0.0

        for attempt in range(MAX_RETRIES):
            seed = 200 + attempt * 777
            generator = torch.Generator(device=DEVICE).manual_seed(seed)

            result = pipe(
                prompt=prompt, negative_prompt=neg,
                width=entry['gen_w'], height=entry['gen_h'],
                num_inference_steps=entry['steps'],
                guidance_scale=entry['guidance'],
                generator=generator,
            )
            img = postprocess_building(result.images[0], entry)

            passed, opaque_pct = quality_check(img, btype)

            if opaque_pct > best_opaque:
                best_opaque = opaque_pct
                best_img = img.copy()

            if passed:
                img.save(str(out_path), 'PNG')
                kb = out_path.stat().st_size / 1024
                print(f'  OK seed={seed} opaque={opaque_pct:.1%} {kb:.0f}KB {time.time()-t0:.1f}s' + (f' (attempt {attempt+1})' if attempt > 0 else ''))
                tracker.mark_done(name)
                free_vram()
                break
            else:
                print(f'  attempt {attempt+1}/{MAX_RETRIES} opaque={opaque_pct:.1%}', end='')
                if attempt < MAX_RETRIES - 1:
                    print(', retrying...')
                else:
                    print(f'\n  Using best attempt (opaque={best_opaque:.1%})')
                    best_img.save(str(out_path), 'PNG')
                    kb = out_path.stat().st_size / 1024
                    print(f'  {kb:.0f}KB {time.time()-t0:.1f}s [BEST OF {MAX_RETRIES}]')
                    tracker.mark_done(name)
                    free_vram()

    print('\nV2 re-generation done!')

No pipeline loaded — nothing to do.


In [7]:
import matplotlib.pyplot as plt

# ─── Final quality scan ───
out_dir = OUTPUT_BASE / 'buildings'
print('=== Final quality scan ===')
final_fails = scan_buildings_quality(out_dir)
if final_fails:
    print(f'\nWARNING: {len(final_fails)} still failing: {final_fails}')
else:
    print('\nAll buildings pass quality check!')

# ─── Preview regenerated items ───
regen_pngs = [out_dir / f'{n}.png' for n in to_regen if (out_dir / f'{n}.png').exists()]
if regen_pngs:
    # Group by type
    type_order = ['castle', 'gate', 'inn', 'shop', 'church', 'region']
    for btype in type_order:
        type_pngs = [p for p in regen_pngs if f'bld_{btype}_' in p.stem]
        if not type_pngs:
            continue
        cols = min(6, len(type_pngs))
        rows = (len(type_pngs) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
        axes = np.atleast_2d(axes)
        for ax in axes.flat:
            ax.axis('off')
        for i, png in enumerate(type_pngs):
            ax = axes[i // cols][i % cols]
            img = Image.open(png)
            ax.imshow(img)
            ax.set_title(png.stem.replace(f'bld_{btype}_', ''), fontsize=7)
        plt.suptitle(f'V2 Regen: {btype} ({len(type_pngs)})', fontsize=12)
        plt.tight_layout()
        plt.show()
else:
    print('Nothing regenerated.')

=== Final quality scan ===
  bld_blacksmith                      64x64   40.8%  bbox=42.5%      4KB  OK
  bld_castle                          64x64   23.3%  bbox=40.9%      3KB  OK
  bld_castle_r1                       320x320   66.7%  bbox=98.4%    159KB  OK
  bld_castle_r10                      320x320   71.4%  bbox=98.1%    152KB  OK
  bld_castle_r11                      320x320   67.9%  bbox=96.9%    148KB  OK
  bld_castle_r12                      320x320   70.7%  bbox=99.7%    163KB  OK
  bld_castle_r2                       320x320   71.8%  bbox=98.8%    160KB  OK
  bld_castle_r3                       320x320   28.9%  bbox=83.4%     70KB  OK
  bld_castle_r4                       320x320   47.0%  bbox=96.2%    113KB  OK
  bld_castle_r5                       320x320   67.2%  bbox=96.6%    145KB  OK
  bld_castle_r6                       320x320   75.1%  bbox=97.2%    166KB  OK
  bld_castle_r7                       320x320   62.0%  bbox=95.6%    157KB  OK
  bld_castle_r8              

In [8]:
import shutil

manifest = {}
d = OUTPUT_BASE / 'buildings'
if d.exists():
    keys = sorted(f.stem for f in d.glob('*.png') if not f.stem.startswith('_'))
    if keys:
        manifest['buildings'] = keys

manifest_path = OUTPUT_BASE / 'manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Manifest:')
for cat, keys in manifest.items():
    print(f'  {cat}: {len(keys)}')

zip_path = '/content/buildings_v2_output'
shutil.make_archive(zip_path, 'zip', str(OUTPUT_BASE))
print(f'\nZip: {os.path.getsize(zip_path + ".zip") / 1024 / 1024:.1f} MB')

if ON_COLAB:
    from google.colab import files
    files.download(f'{zip_path}.zip')
else:
    print(f'Download: {zip_path}.zip')

Manifest:
  buildings: 80

Zip: 7.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>